# Campaign ROAS

Every campaign's spend, revenue and ROAS for one market and period, highest ROAS first. Set the market and the dates in the parameters cell, and choose **Kernel → Restart Kernel and Run All Cells…**.

*Copied the helpers from my old notebook. Works for the US. – Alex*

In [ ]:
import pandas as pd

START, END = "2025-01-01", "2025-12-31"  # 2025
MARKET = "us"

## Helpers

In [ ]:
DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
DAY_FIRST_MONTHS = ["2025-03"]


def read_export(month):
    """Read one month's sessions export, with dates as dates and revenue as numbers."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv")
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


def read_sessions(start, end):
    """Return the clean sessions from start to end, from every month's export they're in."""
    months = [str(month) for month in pd.period_range(start, end, freq="M")]
    raw = pd.concat([read_export(month) for month in months], ignore_index=True)
    deduped = raw.drop_duplicates()
    return deduped[deduped["date"].between(start, end) & (deduped["session_medium"] != "test")].reset_index(drop=True)


def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


def read_spend(start, end):
    """Return the three platforms' spend from start to end, one row per date, platform and campaign."""
    google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
    meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
    youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
    google = google.rename(columns={"campaign_name": "campaign"})
    google["spend"] = google["cost_micros"] / 1_000_000
    google["platform"] = "Google Ads"
    meta = meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Amount spent (USD)": "spend"})
    meta["platform"] = "Meta"
    youtube = youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Cost": "spend"})
    youtube["platform"] = "YouTube"
    columns = ["date", "platform", "campaign", "spend"]
    spend = pd.concat([google[columns], meta[columns], youtube[columns]], ignore_index=True)
    spend = spend[spend["date"].between(start, end)].reset_index(drop=True)
    spend["campaign"] = campaign_key(spend["campaign"])
    return spend.groupby(["date", "platform", "campaign"], as_index=False)["spend"].sum()

## The report

In [ ]:
sessions = read_sessions(START, END)
spend = read_spend(START, END)

revenue = sessions.groupby(["date", "session_campaign"], as_index=False)["revenue"].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = campaign_key(revenue["campaign"])

market_spend = spend[spend["campaign"].str.split("_").str[1] == MARKET]
days = market_spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
campaigns = days.groupby(["platform", "campaign"], as_index=False)[["spend", "revenue"]].sum()
campaigns["roas"] = campaigns["revenue"] / campaigns["spend"]
campaigns.sort_values("roas", ascending=False).round(2)